# 🧠 Master Pipeline: Eye Movement-Based Schizophrenia Recognition

Notebook này chứa toàn bộ quy trình chạy thực nghiệm cho dự án nhận diện tâm thần phân liệt dựa trên dữ liệu chuyển động mắt (scanpaths). 
Hệ thống bao gồm 4 Tiers chính, các mô hình nâng cao (ST-GNN standalone, GNN+CEFAM Hybrid, và BiCA-HS Transformer), cùng quy trình hiệu chuẩn ngưỡng (threshold calibration) và nghiên cứu cắt bỏ (ablation study).

---

## 🛠 Bước 1: Cấu hình Môi trường & Cài đặt Thư viện

Nếu bạn chạy trên **Google Colab**, hãy chuyển đổi Runtime sang GPU để tăng tốc huấn luyện: **Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU**.

In [ ]:
# 1. Clone repository nếu chưa có
# !git clone https://github.com/haimayoi/eye-movement-based-schizophrenia-recognition.git
# %cd eye-movement-based-schizophrenia-recognition

# 2. Cài đặt các thư viện cần thiết
!pip install torch-geometric
!pip install pyyaml omegaconf optuna catboost pyarrow openpyxl xgboost lightgbm scikit-learn pandas numpy matplotlib seaborn

## 📂 Bước 2: Giải nén & Chuẩn bị Dữ liệu

Tải tệp tin `data.zip` (hoặc mount từ Google Drive) và chạy lệnh giải nén để có đầy đủ thư mục `data/` chứa dữ liệu gốc `EMS/`.

In [ ]:
# Mount Google Drive nếu cần thiết
# from google.colab import drive
# drive.mount('/content/drive')

# Giải nén dữ liệu
!unzip -o data.zip -d .

## 🚀 Bước 3: Tier 1 & Tier 2 — Tiền xử lý & Trích xuất Đặc trưng

- **Tier 1 (Preprocessing)**: Làm sạch dữ liệu, xử lý lọc nhiễu không gian/thời gian và sinh parquet.
- **Tier 2 (Feature Engineering)**: Trích xuất đặc trưng mức độ kích thích (stimulus-level) và tổng hợp mức độ đối tượng (subject-level).

In [ ]:
# Sinh bản đồ danh mục hình ảnh (stimulus categories)
!python -m src.utils.generate_category_map

# Chạy lọc nhiễu và làm sạch dữ liệu
!python -m src.tier1_preprocessing.preprocess

# Trích xuất đặc trưng ngữ cảnh đa tầng
!python -m src.tier2_features.stimulus_features

# Tổng hợp đặc trưng lên cấp độ đối tượng
!python -m src.tier2_features.subject_aggregator

## 📊 Bước 4: Tier 3 — Tabular Baselines (XGBoost, LightGBM, CatBoost)

Huấn luyện các bộ phân loại học máy truyền thống trên đặc trưng tổng hợp cấp độ đối tượng để thiết lập baseline.

In [ ]:
# Chạy huấn luyện và đánh giá mô hình học máy dạng bảng
!PYTHONPATH=. python -m src.tier3_tabular.tabular_models

## 🔷 Bước 5: Xây dựng Đồ thị GNN

Xây dựng scanpath graph cho GNN và sinh đặc trưng RINet giả lập (dummy RINet 1056-dim).

In [ ]:
!python -m src.tier4_advanced.graph_builder

## 🔥 Bước 6: Tier 4 — Spatiotemporal GNN Standalone

Mô hình đồ thị không gian-thời gian cơ bản (chỉ gồm GNN stream, không có CEFAM fusion). 

> **Lưu ý Resumption:** Tất cả các kịch bản Tier 4 và BiCA đã được tích hợp cơ chế khôi phục từ checkpoint. Nếu quá trình chạy bị ngắt giữa chừng, khi chạy lại script sẽ phát hiện checkpoint có sẵn (ví dụ: `stgnn_fold_0_best.pt`), tải model để sinh kết quả validation của fold đó và chuyển sang fold tiếp theo.

In [ ]:
# Huấn luyện mô hình ST-GNN Standalone
!PYTHONPATH=. python scripts/train_tier4.py --config configs/stgnn_config.yaml

# Hiệu chuẩn ngưỡng tối ưu cho ST-GNN
!PYTHONPATH=. python scripts/calibrate_threshold.py --preds results/stgnn_subject_val_predictions.csv

## 🔥 Bước 7: Tier 4 — GNN-CEFAM Hybrid Model

Mô hình kết hợp dòng đồ thị không gian-thời gian với đặc trưng tĩnh thủ công thông qua cơ chế CEFAM Fusion.

In [ ]:
# Huấn luyện GNN-CEFAM Hybrid
!PYTHONPATH=. python scripts/train_tier4.py --config configs/cefam_config.yaml

# Hiệu chuẩn ngưỡng tối ưu
!PYTHONPATH=. python scripts/calibrate_threshold.py --preds results/cefam_subject_val_predictions.csv

## 🔥 Bước 8: Tier 4 — BiCA-HS (Transformer Model)

Mô hình Transformer lai ghép đa dòng sử dụng cơ chế Cross-Attention hai chiều.

In [ ]:
# Huấn luyện BiCA-HS
!PYTHONPATH=. python "Bidirectional Cross-Attention Hybrid Stream/scripts/train_bica.py" --config configs/bica_config.yaml

# Sinh kết quả đánh giá K-Fold đầy đủ
!PYTHONPATH=. python "Bidirectional Cross-Attention Hybrid Stream/scripts/evaluate_bica.py" --config configs/bica_config.yaml

# Hiệu chuẩn ngưỡng tối ưu cho BiCA-HS
!PYTHONPATH=. python scripts/calibrate_threshold.py --preds "Bidirectional Cross-Attention Hybrid Stream/results/logs/bica_subject_val_predictions.csv"

## 🧪 Bước 9: Ablation Study (Nghiên cứu Cắt bỏ)

Chạy phân tích chi tiết đóng góp của từng thành phần trong mô hình và vẽ các biểu đồ phân tích xuất bản chất lượng cao.

In [ ]:
# Chạy phân tích so sánh và độ nhạy của mô hình
!python experiments/ablation/run_ablation_analysis.py

# Sinh các hình vẽ trực quan kết quả cắt bỏ
!python experiments/ablation/plot_ablation.py

In [ ]:
# Hiển thị các hình vẽ kết quả trực tiếp trong notebook
from IPython.display import Image, display
import os

fig_dir = "experiments/ablation/figures"
figures = [
    "F1_model_comparison.png",
    "F2_component_contribution.png",
    "F3_category_analysis.png",
    "F5_fold_stability.png",
    "F7_roc_curves.png"
]

for fig in figures:
    path = os.path.join(fig_dir, fig)
    if os.path.exists(path):
        print(f"=== {fig} ===")
        display(Image(filename=path))
        print("\n")

## 📥 Bước 10: Sao lưu Checkpoints & Kết quả về Google Drive

Sao chép tất cả các checkpoints và kết quả từ local runtime về Drive để lưu trữ lâu dài.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 1. Lưu kết quả CEFAM
!mkdir -p "/content/drive/MyDrive/SZ_Recognition_Results/cefam/checkpoints"
!cp -r results/checkpoints/cefam_fold_*.pt "/content/drive/MyDrive/SZ_Recognition_Results/cefam/checkpoints/"
!cp results/cefam_results_summary.json "/content/drive/MyDrive/SZ_Recognition_Results/cefam/"
!cp results/cefam_subject_val_predictions.csv "/content/drive/MyDrive/SZ_Recognition_Results/cefam/"

# 2. Lưu kết quả ST-GNN
!mkdir -p "/content/drive/MyDrive/SZ_Recognition_Results/stgnn/checkpoints"
!cp -r results/checkpoints/stgnn_fold_*.pt "/content/drive/MyDrive/SZ_Recognition_Results/stgnn/checkpoints/"
!cp results/stgnn_results_summary.json "/content/drive/MyDrive/SZ_Recognition_Results/stgnn/"
!cp results/stgnn_subject_val_predictions.csv "/content/drive/MyDrive/SZ_Recognition_Results/stgnn/"

# 3. Lưu kết quả BiCA
!mkdir -p "/content/drive/MyDrive/SZ_Recognition_Results/bica/checkpoints"
!cp -r "Bidirectional Cross-Attention Hybrid Stream/results/checkpoints/"* "/content/drive/MyDrive/SZ_Recognition_Results/bica/checkpoints/"
!cp "Bidirectional Cross-Attention Hybrid Stream/results/logs/"* "/content/drive/MyDrive/SZ_Recognition_Results/bica/"

# 4. Lưu kết quả Ablation Study
!mkdir -p "/content/drive/MyDrive/SZ_Recognition_Results/ablation/figures"
!cp experiments/ablation/results/* "/content/drive/MyDrive/SZ_Recognition_Results/ablation/"
!cp experiments/ablation/figures/* "/content/drive/MyDrive/SZ_Recognition_Results/ablation/figures/"

print("Đã sao lưu toàn bộ kết quả lên Google Drive thành công!")